In [17]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv('../.env')
token = os.getenv('ONEMAP_TOKEN')

# Verify token loaded
print("Token loaded:", token is not None)

df = pd.read_csv('../data/raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')
unique_addresses = df[['block', 'street_name']].drop_duplicates().reset_index(drop=True)

print(f"Total transactions: {len(df):,}")
print(f"Unique addresses:   {len(unique_addresses):,}")

Token loaded: True
Total transactions: 233,055
Unique addresses:   9,712


In [18]:
def geocode_address(block, street_name, token, retries=3):
    query = f"{block} {street_name} SINGAPORE"
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {
        "Authorization": token
    }
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

# Test on one address
block, street = unique_addresses.iloc[0]['block'], unique_addresses.iloc[0]['street_name']
lat, lon = geocode_address(block, street, token)
print(f"Test address: {block} {street}")
print(f"Coordinates:  {lat}, {lon}")

Test address: 406 ANG MO KIO AVE 10
Coordinates:  1.36200453938712, 103.853879910407


In [19]:
test = unique_addresses.head(50).copy().reset_index(drop=True)
results = []

for idx, row in test.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

test_results = pd.DataFrame(results)
print(f"Success: {test_results['latitude'].notna().sum()} / 50")
print(f"Failed:  {test_results['latitude'].isna().sum()} / 50")

Success: 50 / 50
Failed:  0 / 50


In [22]:
results = []

for idx, row in unique_addresses.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    
    if (idx + 1) % 500 == 0:
        print(f"Processed {idx + 1:,} / {len(unique_addresses):,}")
        pd.DataFrame(results).to_csv('../data/processed/geocoded_addresses.csv', index=False)
    
    time.sleep(0.25)

geocoded = pd.DataFrame(results)
geocoded.to_csv('../data/processed/geocoded_addresses.csv', index=False)
print(f"Done. Geocoded {len(geocoded):,} addresses.")

Processed 500 / 9,712
Processed 1,000 / 9,712
Processed 1,500 / 9,712
Processed 2,000 / 9,712
Processed 2,500 / 9,712
Processed 3,000 / 9,712
Processed 3,500 / 9,712
Processed 4,000 / 9,712
Processed 4,500 / 9,712
Processed 5,000 / 9,712
Processed 5,500 / 9,712
Processed 6,000 / 9,712
Processed 6,500 / 9,712
Processed 7,000 / 9,712
Processed 7,500 / 9,712
Processed 8,000 / 9,712
Processed 8,500 / 9,712
Processed 9,000 / 9,712
Processed 9,500 / 9,712
Done. Geocoded 9,712 addresses.


In [24]:
print(f"Failed: {geocoded['latitude'].isna().sum()}")


Failed: 95


In [25]:
failed = geocoded[geocoded['latitude'].isna()]
print(f"Failed: {len(failed)}")
print(failed.head(10).to_string())

Failed: 95
    block        street_name  latitude  longitude
269    32         NEW MKT RD       NaN        NaN
335   411  C'WEALTH AVE WEST       NaN        NaN
568     3    ST. GEORGE'S RD       NaN        NaN
583    12       FARRER PK RD       NaN        NaN
592    21    ST. GEORGE'S RD       NaN        NaN
686    83        C'WEALTH CL       NaN        NaN
689    81        C'WEALTH CL       NaN        NaN
691   109      C'WEALTH CRES       NaN        NaN
692    94        C'WEALTH DR       NaN        NaN
693   111      C'WEALTH CRES       NaN        NaN


In [26]:
failed_addresses = geocoded[geocoded['latitude'].isna()][['block', 'street_name']]
affected = df.merge(failed_addresses, on=['block', 'street_name'])
print(f"Transactions affected: {len(affected):,}")
print(f"As % of dataset: {len(affected)/len(df)*100:.1f}%")

Transactions affected: 2,353
As % of dataset: 1.0%


In [31]:
abbreviations = {
    "C'WEALTH": "COMMONWEALTH",
    "ST.": "SAINT",
    "NEW MKT RD": "NEW MARKET ROAD",
    "FARRER PK RD": "FARRER PARK ROAD",
    "SPOTTISWOODE PK RD": "SPOTTISWOODE PARK ROAD",
    "EVERTON PK": "EVERTON PARK",
    "BIDADARI PK DR": "BIDADARI PARK DRIVE",
    " PK ": " PARK ",
    "AVE": "AVENUE",
    "RD": "ROAD",
    "CL": "CLOSE",
    "CRES": "CRESCENT",
    "DR": "DRIVE",
    "ST ": "STREET ",
    "BT ": "BUKIT ",
    "UPP ": "UPPER ",
    "KG ": "KAMPONG ",
}

def expand_abbreviations(street_name):
    for abbrev, full in abbreviations.items():
        street_name = street_name.replace(abbrev, full)
    return street_name

# Test on known failures
test_cases = ["SPOTTISWOODE PK RD", "EVERTON PK", "BIDADARI PK DR", "C'WEALTH AVE WEST", "ST. GEORGE'S RD"]
for t in test_cases:
    print(f"{t} -> {expand_abbreviations(t)}")

SPOTTISWOODE PK RD -> SPOTTISWOODE PARK ROAD
EVERTON PK -> EVERTON PARK
BIDADARI PK DR -> BIDADARI PARK DRIVEIVE
C'WEALTH AVE WEST -> COMMONWEALTH AVENUE WEST
ST. GEORGE'S RD -> SAINT GEORGE'S ROAD


In [32]:
retry_results = []

for idx, row in failed_addresses.iterrows():
    expanded_street = expand_abbreviations(row['street_name'])
    lat, lon = geocode_address(row['block'], expanded_street, token)
    retry_results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

retry_df = pd.DataFrame(retry_results)
print(f"Retry success: {retry_df['latitude'].notna().sum()} / {len(retry_df)}")
print(f"Still failed: {retry_df['latitude'].isna().sum()}")

Retry success: 85 / 95
Still failed: 10


In [33]:
# Update geocoded with successful retry results
for _, row in retry_df[retry_df['latitude'].notna()].iterrows():
    mask = (geocoded['block'] == row['block']) & (geocoded['street_name'] == row['street_name'])
    geocoded.loc[mask, 'latitude'] = row['latitude']
    geocoded.loc[mask, 'longitude'] = row['longitude']

print(f"Failed after retry: {geocoded['latitude'].isna().sum()}")

# Save updated geocoded addresses
geocoded.to_csv('../data/processed/geocoded_addresses.csv', index=False)
print("Saved.")

Failed after retry: 10
Saved.


In [34]:
still_failed = geocoded[geocoded['latitude'].isna()]
print(still_failed[['block', 'street_name']].to_string())

     block     street_name
9603  106B  BIDADARI PK DR
9604  105A  BIDADARI PK DR
9605  106A  BIDADARI PK DR
9606  105B  BIDADARI PK DR
9661  102B  BIDADARI PK DR
9662  101A  BIDADARI PK DR
9663  102A  BIDADARI PK DR
9708  103B  BIDADARI PK DR
9709  103A  BIDADARI PK DR
9710  104A  BIDADARI PK DR


In [35]:
nbidadari = geocoded[geocoded['street_name'] == 'BIDADARI PK DR']
affected = df[df['street_name'] == 'BIDADARI PK DR']
print(f"Bidadari transactions: {len(affected):,}")


Bidadari transactions: 205
